# Two-track ablation figure

Rows = ablations, columns = judge models, one column block per data track. Each
cell is the change the ablation made to one judge's score, in points, with the
post-ablation value beneath it and `*` for significant at 95% (corpus-level
paired bootstrap, n=100,000). The strip on top is the un-ablated baseline the
whole block is measured against.

Reads the `unified_*.json` chart data `merge_ablation_runs.py` writes — no merge
re-run, no bootstrap here. Writes a PDF (vector, for the paper) and a PNG (shown
inline below) to `output/figures/<FIGURE>/`.

What every figure has to agree on — where the chart data lives, which cells are
artefacts, the delta palette — comes from `common.py`, so a figure drawn here
and one drawn by a script cannot disagree about the numbers they may show.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src").is_dir() and (p / "output").is_dir())
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.figures.common import (
    PDF_RCPARAMS,
    UNSUPPORTED_CELLS,
    chart_data_path,
    delta_cmap,
)

VARIANT = "filtered"              # which accuracy report: filtered | unfiltered
COL_AXIS_LABEL = "Judge model"

# Shorter judge names for the column headers.
MODEL_LABELS = {"claude-sonnet-4-5": "sonnet-4-5",
                "claude-opus-4-5": "opus-4-5",
                "claude-opus-4-6": "opus-4-6"}
print(ROOT)

## Which figure

Run **one** of the next two cells — each is one paper figure, and the second
overrides the first. Everything that decides what the figure says is here:
nothing below this point needs editing to draw a different experiment.

In [ ]:
# --- Controlled changes: novelty criterion, prior work, reasoning, idea format
FIGURE = "controlled-study"       # output subdirectory and filename prefix

# One merge output dir per track: (block title, dir under output/ablation_sweeps,
# name token). The token fills {track} in a ROWS entry below.
TRACKS = [
    ("Human-only", "merge_hvh", "human"),
    ("Human + generated", "merge_vanilla", "vanilla"),
]

# (panel name, row label), top to bottom. {setup} and {track} expand per figure:
# the plan-form ablation runs on a different instance set per track, so its panel
# is named per track, while the others are named the same in both.
ROWS = [
    ("vague_criterion", "Vague novelty criterion"),
    ("no_criteria_prompt", "No criterion in prompt"),
    ("retrieval", "+ Related work (retrieval)"),
    ("low_judge_reasoning", "Reasoning effort = low"),
    ("convert_to_plan_form_{setup}_{track}", "Idea as research plan"),
]

# Judge columns, left to right.
MODELS = ["claude-sonnet-4-5", "claude-opus-4-5", "claude-opus-4-6",
          "gpt-5.1", "gpt-5.2", "gpt-5.4"]

# The two setups score different things, so each names its own metric. The metric
# is in the filename, not on the figure — the caption has to say which it is.
METRICS = {"pairwise": "pairwise_accuracy_strict", "pointwise": "recall_neg"}

ROW_AXIS_LABEL = "Ablation"

In [ ]:
# --- Prompt design case study: three wordings of the same AI-researcher prompt.
# Pairwise only — these runs have no pointwise data.
FIGURE = "prompt-design"

TRACKS = [
    ("Human-only", "merge_ai_researcher_hvh-plan", "human"),
    ("Human + generated", "merge_ai_researcher_vanilla_ai-plan", "vanilla"),
]

ROWS = [
    ("ai_researcher_no_review", "No review mention"),
    ("ai_researcher_comparative", "Comparative language"),
]

MODELS = ["claude-opus-4-6", "gpt-5.4"]

METRICS = {"pairwise": "pairwise_accuracy"}

ROW_AXIS_LABEL = "Prompt variant"

## The numbers

`table(setup)` is the whole data path: one tidy row per (track, ablation, judge),
everything already in points. Print it, sort it, `.query()` it — if a cell in the
figure looks wrong, the reason is visible here, before any drawing happens.

A missing panel, or a cell `common.UNSUPPORTED_CELLS` blanks (a judge that cannot
honour the ablation), gives `NaN` and draws as `—`.

In [ ]:
def panels(merge_dir, setup):
    """The panel dict of one merge's saved chart data."""
    path = chart_data_path(ROOT / "output/ablation_sweeps" / merge_dir,
                           setup, VARIANT)
    return json.loads(path.read_text())["panels"]


def table(setup):
    """One row per (track, ablation, judge). Scores and deltas are in points."""
    metric = METRICS[setup]
    recs = []
    for title, merge_dir, token in TRACKS:
        ps = panels(merge_dir, setup)
        for name, label in ROWS:
            panel = ps.get(name.format(setup=setup, track=token))
            for model in MODELS:
                if panel is None or (name, model) in UNSUPPORTED_CELLS:
                    base = abl = sig = None
                else:
                    base = panel["baseline_metrics"].get(model, {}).get(metric)
                    abl = panel["ablation_metrics"].get(model, {}).get(metric)
                    # Bootstrap results are keyed "<model>||<metric>" on disk.
                    sig = panel["bootstrap_results"].get(
                        f"{model}||{metric}", {}).get("significant")
                recs.append({
                    "track": title,
                    "ablation": label,
                    "judge": MODEL_LABELS.get(model, model),
                    "baseline": None if base is None else base * 100,
                    "ablated": None if abl is None else abl * 100,
                    "delta": None if base is None or abl is None
                             else (abl - base) * 100,
                    "significant": bool(sig),
                })
    return pd.DataFrame(recs)

In [ ]:
print(table(next(iter(METRICS))).to_string())

## The figure

One `pcolormesh` per track, then the cell text on top of it. Diverging red→green
scale centred on zero and shared across both blocks, so a colour means the same
thing left and right; every cell also prints its number, so colour is never the
only encoding. The baseline strip is drawn outside the grid, in a shade of its
own, so it never reads as a delta of zero.

The layout is an explicit inch budget rather than a layout engine: every gap on
the page is one named number, so a figure that looks wrong is one edit away.

In [ ]:
CMAP = delta_cmap()               # red → off-white → green; grey for no number
BASELINE_BG = "#f4f6f8"           # the baseline strip, a shade of its own
plt.rcParams.update({"font.family": "sans-serif", **PDF_RCPARAMS})

# Layout in inches, so a cell is the size it says it is. Every gap on the page
# is one number here: no layout engine to reverse-engineer when it looks wrong.
CELL_IN = 0.44                    # height of one ablation row
STRIP_ROWS = 0.56                 # baseline strip height, in rows — shorter than
                                  # a data row, so it reads as a header
GUTTER_IN = 0.17                  # between the two blocks
MARGIN_IN = {"left": 1.51,        # room for the row names and the axis title
             "right": 0.05, "top": 0.35, "bottom": 0.61}


def draw(setup, width=7.1):
    """Draw one setup's figure, save PDF + PNG, return the PNG path."""
    df = table(setup)
    labels = [label for _, label in ROWS]
    judges = [MODEL_LABELS.get(m, m) for m in MODELS]
    vmax = max(1.0, np.nanmax(np.abs(df["delta"].astype(float))))

    grid_in = CELL_IN * (len(labels) + STRIP_ROWS)
    fig_h = grid_in + MARGIN_IN["top"] + MARGIN_IN["bottom"]
    blocks_in = width - MARGIN_IN["left"] - MARGIN_IN["right"]
    block_in = (blocks_in - GUTTER_IN * (len(TRACKS) - 1)) / len(TRACKS)

    fig = plt.figure(figsize=(width, fig_h))
    gs = fig.add_gridspec(
        1, len(TRACKS), wspace=GUTTER_IN / block_in,
        left=MARGIN_IN["left"] / width, right=1 - MARGIN_IN["right"] / width,
        top=1 - MARGIN_IN["top"] / fig_h, bottom=MARGIN_IN["bottom"] / fig_h)
    axes = [fig.add_subplot(gs[0, i]) for i in range(len(TRACKS))]

    for ax, (title, _, _) in zip(axes, TRACKS):
        block = df[df["track"] == title].set_index(["ablation", "judge"])
        grid = lambda col: block[col].unstack().reindex(index=labels, columns=judges)
        delta, ablated, sig = grid("delta"), grid("ablated"), grid("significant")
        # Every ablation in a track shares one baseline run, so it is one strip
        # above the grid rather than a number repeated down every row.
        base = grid("baseline").bfill().iloc[0]

        # The mesh holds the ablation rows only. The baseline strip is drawn
        # separately, in its own shade, so it never reads as a delta of zero.
        ax.pcolormesh(np.arange(len(judges) + 1), np.arange(1, len(labels) + 2),
                      np.ma.masked_invalid(delta.to_numpy(float)), cmap=CMAP,
                      vmin=-vmax, vmax=vmax, edgecolors="white", linewidth=1.2)
        strip_top = 1 - STRIP_ROWS
        ax.axhspan(strip_top, 0.995, facecolor=BASELINE_BG, zorder=0)
        ax.vlines(range(1, len(judges)), strip_top, 0.995,
                  color="white", linewidth=1.2)
        ax.set_xlim(0, len(judges))
        ax.set_ylim(len(labels) + 1, strip_top)      # baseline strip on top

        for j, judge in enumerate(judges):
            ax.text(j + 0.5, (strip_top + 1) / 2, f"{base[judge]:.1f}",
                    ha="center", va="center", fontsize=6.4, fontweight="bold",
                    color="#333333")
            for i, label in enumerate(labels):
                d = delta.loc[label, judge]
                if not np.isfinite(float(d if d is not None else np.nan)):
                    ax.text(j + 0.5, i + 1.5, "—", ha="center", va="center",
                            fontsize=6, color="#999999")
                    continue
                # White text only on the dark ends of the scale.
                shade = (d + vmax) / (2 * vmax)
                colour = "white" if shade < 0.25 or shade > 0.82 else "#1a1a1a"
                star = "*" if sig.loc[label, judge] else ""
                ax.text(j + 0.5, i + 1.32, f"{d:+.1f}{star}", ha="center",
                        va="center", fontsize=6.6, fontweight="bold", color=colour)
                # Italic, so the absolute never reads as a second delta.
                ax.text(j + 0.5, i + 1.68, f"({ablated.loc[label, judge]:.1f})",
                        ha="center", va="center", fontsize=5.4, style="italic",
                        color=colour, alpha=0.72)

        ax.set_xticks(np.arange(len(judges)) + 0.5, judges, fontsize=6.2,
                      rotation=38, ha="right", rotation_mode="anchor")
        ax.set_yticks([(strip_top + 1) / 2] + [i + 1.5 for i in range(len(labels))],
                      ["baseline"] + labels, fontsize=6.6)
        ax.set_title(title, fontsize=8, fontweight="bold", pad=8.5,
                     bbox=dict(boxstyle="round,pad=0.35", fc="#eef1f5", ec="none"))
        ax.tick_params(length=0, pad=2)
        for spine in ax.spines.values():
            spine.set_visible(False)

    # The row names, and both axis titles, belong to the left-hand block only.
    for ax in axes[1:]:
        ax.tick_params(labelleft=False)
    # The strip is labelled like a row, but lighter: it is not an ablation.
    axes[0].get_yticklabels()[0].set(style="italic", color="#555555")
    axes[0].set_ylabel(ROW_AXIS_LABEL, fontsize=7.2, fontweight="bold",
                       color="#333333", labelpad=5)
    fig.text((MARGIN_IN["left"] + blocks_in / 2) / width, 0.12 / fig_h,
             COL_AXIS_LABEL, ha="center", va="bottom", fontsize=7.2,
             fontweight="bold", color="#333333")

    out_dir = ROOT / "output/figures" / FIGURE
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = f"{FIGURE}_{setup}_{VARIANT}_{METRICS[setup]}"
    fig.savefig(out_dir / f"{stem}.pdf")
    fig.savefig(out_dir / f"{stem}.png", dpi=300)
    plt.close(fig)
    return out_dir / f"{stem}.png"

In [ ]:
for setup in METRICS:
    png = draw(setup)
    print(png.relative_to(ROOT), "(+ .pdf)")
    display(Image(filename=str(png)))